# LangGraph 003 — What Is Agentic AI?

Two of the ideas in this lesson, as code you can change: **planning as search**
and **controlling autonomy**. Then a short exercise in recognising agentic
systems. Plain Python — **no model, no API key**.

| Part | What we check |
|---|---|
| A | five plans: three ruled out for three different reasons, the best of two chosen |
| B | one gate applies permissions, approval, override and guardrails |
| C | score six systems against the six characteristics |

## Part A — Planning: generate, evaluate, select

In [ ]:
# A goal, kept in the agent's memory as data
goal = {
    "main_goal": "hire a backend engineer",
    "constraints": {"location": "remote", "experience": "2-4 years", "budget": 50000},
    "status": "planning",
    "progress": [],
}

TOOLS_AVAILABLE = {"linkedin_api", "resume_parser", "calendar", "mail"}

# Step 1 - several candidate plans, not one
plans = {
    "A: post on job sites": {"needs": {"linkedin_api", "resume_parser"},
                             "days": 21, "cost": 20000, "risk": 0.3, "fits_remote": True},
    "B: hiring agency":     {"needs": {"mail"},
                             "days": 14, "cost": 90000, "risk": 0.2, "fits_remote": True},
    "C: web search + cold email": {"needs": {"google_search", "mail"},
                             "days": 30, "cost": 5000, "risk": 0.5, "fits_remote": True},
    "D: internal referrals": {"needs": {"mail"},
                             "days": 25, "cost": 10000, "risk": 0.6, "fits_remote": True},
    "E: walk-in interview day": {"needs": {"calendar"},
                             "days": 7, "cost": 15000, "risk": 0.4, "fits_remote": False},
}

# Step 2 - evaluate: drop plans that break a hard rule, score the rest
def evaluate(name, p):
    if not p["needs"] <= TOOLS_AVAILABLE:
        return None, f"needs {sorted(p['needs'] - TOOLS_AVAILABLE)}, which we do not have"
    if p["cost"] > goal["constraints"]["budget"]:
        return None, "over budget"
    if goal["constraints"]["location"] == "remote" and not p["fits_remote"]:
        return None, "does not fit a remote hire"
    return p["days"] + 30 * p["risk"], "ok"          # lower is better

for name, p in plans.items():
    score, why = evaluate(name, p)
    print(f"{name:<28} {why if score is None else f'score {score:.0f}'}")

# Step 3 - select (a policy here; could be a person instead)
scored = {n: evaluate(n, p)[0] for n, p in plans.items()}
best = min((n for n in scored if scored[n] is not None), key=scored.get)
goal["status"], goal["plan"] = "executing", best
print("selected:", best)

In [ ]:
assert best == "A: post on job sites"
assert sum(s is None for s in scored.values()) == 3
print("ruled out:", [n for n, s in scored.items() if s is None])

The numbers are invented to show the process. Now change the goal and plan
again — a goal can change half-way, and that sends the agent back to planning.

In [ ]:
goal["constraints"]["budget"] = 100_000           # the agency is now affordable
TOOLS_AVAILABLE.add("google_search")              # and web search is possible
scored = {n: evaluate(n, p)[0] for n, p in plans.items()}
best = min((n for n in scored if scored[n] is not None), key=scored.get)
print({n: s for n, s in scored.items()})
print("new plan:", best)
assert best == "B: hiring agency"

## Part B — Controlling autonomy

In [ ]:
# Four ways to control autonomy, as one gate every action passes through.
from datetime import date

PERMISSIONS = {"screen_resume", "draft_jd", "schedule_interview", "send_email"}  # scope
NEEDS_APPROVAL = {"post_jd", "send_offer", "reject_candidate", "run_ads"}       # human in the loop
paused = False                                                                  # override

def guardrails(action, args):                                                   # hard rules
    if action == "schedule_interview" and args["day"].weekday() >= 5:
        return "never schedule interviews at the weekend"
    if action == "send_offer" and args["salary"] > 2_400_000:
        return "salary above the approved band"
    return None

def gate(action, args, approve):
    if paused:
        return "blocked: agent is paused"
    broken = guardrails(action, args)
    if broken:
        return f"blocked: {broken}"
    if action in NEEDS_APPROVAL:
        return "done (approved)" if approve(action, args) else "held: person said no"
    if action in PERMISSIONS:
        return "done"
    return "blocked: outside the agent's permissions"

say_yes = lambda action, args: True
requests = [
    ("screen_resume", {}),
    ("schedule_interview", {"day": date(2026, 9, 26)}),      # a Saturday
    ("schedule_interview", {"day": date(2026, 9, 28)}),      # a Monday
    ("send_offer", {"salary": 3_000_000}),
    ("send_offer", {"salary": 1_800_000}),
    ("delete_job_posting", {}),
]
for action, args in requests:
    print(f"{action:<20} -> {gate(action, args, say_yes)}")

paused = True
print(f"{'screen_resume':<20} -> {gate('screen_resume', {}, say_yes)}")

In [ ]:
paused = False
say_no = lambda action, args: False
assert gate("send_offer", {"salary": 1_800_000}, say_no) == "held: person said no"
assert gate("delete_job_posting", {}, say_yes).startswith("blocked")
print("approval, permissions, guardrails and override all enforced")

## Part C — Is it agentic?

Mark each system against the six characteristics. A system is agentic when it
has all six; most real products have some.

In [ ]:
TRAITS = ["autonomy", "goal", "planning", "reasoning", "adaptability", "context"]
SYSTEMS = {
    "spam filter":                         [0, 0, 0, 0, 0, 0],
    "chatbot answering one question":      [0, 0, 0, 1, 0, 0],
    "chatbot with memory of the chat":     [0, 0, 0, 1, 0, 1],
    "RAG chatbot with tools":              [0, 0, 0, 1, 0, 1],
    "cron job that posts a JD every week": [1, 0, 0, 0, 0, 0],
    "hiring agent of lesson 002":          [1, 1, 1, 1, 1, 1],
}
for name, marks in SYSTEMS.items():
    missing = [t for t, m in zip(TRAITS, marks) if not m]
    verdict = "agentic" if not missing else f"missing {len(missing)}: {', '.join(missing)}"
    print(f"{name:<37} {verdict}")

The cron job is autonomous — it acts without being asked — but it has no goal,
no plan and cannot adapt. Autonomy alone is not agency.

## What to take away

- Agentic AI takes a **goal** and plans, acts and adapts with little guidance.
- Six characteristics: autonomy, goal, planning, reasoning, adaptability, context.
- Autonomy needs **controls**: permissions, approval, override, guardrails.
- Five parts: brain, orchestrator, tools, memory, supervisor — LangGraph is the
  orchestrator.

## Exercises

1. Add a guardrail: offers may not be sent on the same day as the interview.
2. Give each plan a `needs_approval` flag and make the selection ask a person
   when the best two scores are within 5 of each other.
3. Would you mark your phone's voice assistant as agentic? Fill in its row.